# 산불 진화 접근성 종합 EDA

산불 발생 지점별로 임도, 소방서, 소방용수시설, 산불소화시설, 지형 조건을 함께 붙여 **진화 접근성 취약도**를 분석합니다.

핵심 질문:
- 임도와 소방서가 모두 먼 산불 지점은 어디인가?
- 임도 접근은 가능하지만 소방용수가 부족한 지역은 어디인가?
- 경사가 높고 고도가 높으면서 진화 인프라도 먼 산불 지점은 어디인가?
- 종합 점수 기준으로 우선 관리해야 할 접근 취약 지역은 어디인가?

주의: 별도 GIS 패키지 없이 실행되도록 위경도를 미터 좌표로 근사 변환해 거리 계산합니다. 임도는 `LINESTRING` 좌표점을 샘플링해 최근접 거리를 근사 계산합니다.

In [ ]:
from pathlib import Path
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams["font.family"] = ["Malgun Gothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="Malgun Gothic")

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA_DIR = ROOT / "data" / "processed"
OUT_DIR = ROOT / "outputs" / "integrated_accessibility_eda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIRE_PATH = DATA_DIR / "산불발생위치도_지형특성계산.csv"
ROAD_PATH = DATA_DIR / "전국_임도망도.csv"
STATION_PATH = DATA_DIR / "전국_소방서_좌표.csv"
WATER_PATH = DATA_DIR / "전국_소방용수시설.csv"
FOREST_EXT_PATH = DATA_DIR / "전국_산불소화시설.csv"

## 1. 데이터 로드 및 좌표 정리

In [ ]:
fires = pd.read_csv(FIRE_PATH)
roads = pd.read_csv(ROAD_PATH)
stations = pd.read_csv(STATION_PATH)
water = pd.read_csv(WATER_PATH)
forest_ext = pd.read_csv(FOREST_EXT_PATH)

def filter_korea_bbox(df, lat_col="위도", lon_col="경도"):
    return df[df[lat_col].between(32, 39.5) & df[lon_col].between(124, 132)].copy()

fires = filter_korea_bbox(fires)
stations = filter_korea_bbox(stations)
water = filter_korea_bbox(water)
forest_ext = filter_korea_bbox(forest_ext)

print("산불:", fires.shape)
print("임도:", roads.shape)
print("소방서/안전센터:", stations.shape)
print("소방용수시설:", water.shape)
print("산불소화시설:", forest_ext.shape)

display(fires.head(2))
display(stations.head(2))
display(water.head(2))
display(forest_ext.head(2))

## 2. 거리 계산 함수

점 데이터는 격자 인덱스를 사용해 최근접 시설을 찾습니다. 임도는 `LINESTRING` 좌표를 샘플링한 점 집합으로 변환한 뒤 같은 방식으로 계산합니다.

In [ ]:
def lonlat_to_meter(lon, lat, lat0=36.5):
    radius = 6_371_000
    x = np.deg2rad(lon) * radius * np.cos(np.deg2rad(lat0))
    y = np.deg2rad(lat) * radius
    return x, y

def nearest_distances(source_lon, source_lat, target_lon, target_lat, cell_size=10_000, max_radius_cells=30):
    target_x, target_y = lonlat_to_meter(np.asarray(target_lon), np.asarray(target_lat))
    source_x, source_y = lonlat_to_meter(np.asarray(source_lon), np.asarray(source_lat))

    target_ix = np.floor(target_x / cell_size).astype(int)
    target_iy = np.floor(target_y / cell_size).astype(int)
    grid = {}
    for idx, key in enumerate(zip(target_ix, target_iy)):
        grid.setdefault(key, []).append(idx)

    nearest = np.empty(len(source_x), dtype=float)
    for i, (x, y) in enumerate(zip(source_x, source_y)):
        cx = int(math.floor(x / cell_size))
        cy = int(math.floor(y / cell_size))
        candidates = []

        for radius in range(max_radius_cells + 1):
            for gx in range(cx - radius, cx + radius + 1):
                for gy in range(cy - radius, cy + radius + 1):
                    if radius == 0 or gx in (cx - radius, cx + radius) or gy in (cy - radius, cy + radius):
                        candidates.extend(grid.get((gx, gy), []))
            if candidates:
                dx = target_x[candidates] - x
                dy = target_y[candidates] - y
                nearest[i] = np.sqrt(dx * dx + dy * dy).min()
                break
        else:
            nearest[i] = np.nan

    return nearest / 1000

coord_pattern = re.compile(r"([0-9]+\.[0-9]+) ([0-9]+\.[0-9]+)")

def sample_road_points(road_df, step=10):
    lons = []
    lats = []
    for wkt in road_df["공간좌표"].dropna():
        points = coord_pattern.findall(str(wkt))
        if len(points) > step:
            points = points[::step]
        for lon, lat in points:
            lons.append(float(lon))
            lats.append(float(lat))
    return pd.DataFrame({"경도": lons, "위도": lats})

## 3. 산불 지점별 최근접 인프라 거리 계산

In [ ]:
road_points = sample_road_points(roads, step=10)
print("샘플링된 임도 좌표점:", len(road_points))

access = fires.copy()
access["임도거리_km"] = nearest_distances(access["경도"], access["위도"], road_points["경도"], road_points["위도"])
access["소방서거리_km"] = nearest_distances(access["경도"], access["위도"], stations["경도"], stations["위도"])
access["소방용수거리_km"] = nearest_distances(access["경도"], access["위도"], water["경도"], water["위도"])
access["산불소화시설거리_km"] = nearest_distances(access["경도"], access["위도"], forest_ext["경도"], forest_ext["위도"])

distance_cols = ["임도거리_km", "소방서거리_km", "소방용수거리_km", "산불소화시설거리_km"]
access[["fire_id", "위도", "경도"] + distance_cols].head()

In [ ]:
distance_summary = access[distance_cols].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).T
distance_summary.round(2)

## 4. 거리 분포와 인프라 조합 분석

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
colors = ["#ef4444", "#3b82f6", "#14b8a6", "#f97316"]

for ax, col, color in zip(axes.ravel(), distance_cols, colors):
    sns.histplot(access[col].dropna(), bins=45, kde=True, ax=ax, color=color)
    ax.set_title(col.replace("_km", " 분포"))
    ax.set_xlabel("거리(km)")
    ax.set_ylabel("산불 발생 건수")

plt.tight_layout()
plt.savefig(OUT_DIR / "infrastructure_distance_histograms.png", dpi=160)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.scatterplot(data=access, x="임도거리_km", y="소방서거리_km", s=8, alpha=0.35, ax=axes[0], color="#7c3aed")
axes[0].set_title("임도거리 vs 소방서거리")

sns.scatterplot(data=access, x="임도거리_km", y="소방용수거리_km", s=8, alpha=0.35, ax=axes[1], color="#0891b2")
axes[1].set_title("임도거리 vs 소방용수거리")

sns.scatterplot(data=access, x="임도거리_km", y="경사도(도)", s=8, alpha=0.35, ax=axes[2], color="#dc2626")
axes[2].set_title("임도거리 vs 경사도")

for ax in axes:
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig(OUT_DIR / "accessibility_pair_scatter.png", dpi=160)
plt.show()

## 5. 진화 접근성 취약 점수 만들기

거리와 지형 조건을 0~100 점수로 정규화합니다. 점수가 높을수록 접근 취약성이 크다는 뜻입니다.

가중치 예시:
- 임도거리 30%
- 소방서거리 20%
- 소방용수거리 20%
- 산불소화시설거리 15%
- 경사도 10%
- 고도 5%

In [ ]:
def cap_score(series, cap):
    return (series.clip(lower=0, upper=cap) / cap * 100).fillna(100)

access["임도취약점수"] = cap_score(access["임도거리_km"], 20)
access["소방서취약점수"] = cap_score(access["소방서거리_km"], 30)
access["소방용수취약점수"] = cap_score(access["소방용수거리_km"], 10)
access["산불소화시설취약점수"] = cap_score(access["산불소화시설거리_km"], 30)
access["경사취약점수"] = cap_score(access["경사도(도)"], 30)
access["고도취약점수"] = cap_score(access["고도(m)"], 500)

access["진화접근취약점수"] = (
    access["임도취약점수"] * 0.30
    + access["소방서취약점수"] * 0.20
    + access["소방용수취약점수"] * 0.20
    + access["산불소화시설취약점수"] * 0.15
    + access["경사취약점수"] * 0.10
    + access["고도취약점수"] * 0.05
)

access["접근취약등급"] = pd.cut(
    access["진화접근취약점수"],
    bins=[0, 35, 55, 70, 100],
    labels=["낮음", "보통", "높음", "매우 높음"],
    include_lowest=True,
)

score_cols = [
    "임도취약점수", "소방서취약점수", "소방용수취약점수", "산불소화시설취약점수",
    "경사취약점수", "고도취약점수", "진화접근취약점수", "접근취약등급"
]
access[["fire_id", "위도", "경도"] + distance_cols + score_cols].head()

In [ ]:
grade_summary = (
    access["접근취약등급"]
    .value_counts()
    .reindex(["낮음", "보통", "높음", "매우 높음"])
    .rename_axis("접근취약등급")
    .reset_index(name="산불건수")
)
grade_summary["비율_%"] = grade_summary["산불건수"] / grade_summary["산불건수"].sum() * 100

display(access["진화접근취약점수"].describe().round(2))
display(grade_summary.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

sns.histplot(access["진화접근취약점수"], bins=40, kde=True, ax=axes[0], color="#be123c")
axes[0].set_title("진화 접근 취약 점수 분포")
axes[0].set_xlabel("취약 점수")
axes[0].set_ylabel("산불 발생 건수")

sns.barplot(data=grade_summary, x="접근취약등급", y="산불건수", ax=axes[1], color="#f97316")
axes[1].set_title("접근 취약 등급별 산불 발생 건수")
axes[1].set_xlabel("접근 취약 등급")
axes[1].set_ylabel("산불 발생 건수")
for i, row in grade_summary.iterrows():
    axes[1].text(i, row["산불건수"], f"{row['비율_%']:.1f}%", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "integrated_accessibility_score.png", dpi=160)
plt.show()

## 6. 접근 취약 산불 지점 Top 100

In [ ]:
top_vulnerable = (
    access.sort_values("진화접근취약점수", ascending=False)
    [[
        "fire_id", "위도", "경도", "진화접근취약점수", "접근취약등급",
        "임도거리_km", "소방서거리_km", "소방용수거리_km", "산불소화시설거리_km",
        "고도(m)", "경사도(도)", "TPI(지형위치지수)", "TWI(지형다습지수)"
    ]]
    .head(100)
)

top_vulnerable.to_csv(OUT_DIR / "top_100_fire_suppression_access_vulnerable.csv", index=False, encoding="utf-8-sig")
access.to_csv(OUT_DIR / "fire_integrated_accessibility_scores.csv", index=False, encoding="utf-8-sig")
distance_summary.to_csv(OUT_DIR / "integrated_distance_summary.csv", encoding="utf-8-sig")
grade_summary.to_csv(OUT_DIR / "integrated_accessibility_grade_summary.csv", index=False, encoding="utf-8-sig")

top_vulnerable.round(2)

## 7. 접근 취약 지역 격자 분석

산불 지점을 0.5도 격자로 묶고 평균 취약 점수와 산불 건수를 함께 봅니다.

In [ ]:
grid_size_deg = 0.5
access["lon_grid"] = np.floor(access["경도"] / grid_size_deg) * grid_size_deg
access["lat_grid"] = np.floor(access["위도"] / grid_size_deg) * grid_size_deg

grid_summary = (
    access.groupby(["lat_grid", "lon_grid"])
    .agg(
        산불건수=("fire_id", "count"),
        평균취약점수=("진화접근취약점수", "mean"),
        중앙취약점수=("진화접근취약점수", "median"),
        평균임도거리_km=("임도거리_km", "mean"),
        평균소방서거리_km=("소방서거리_km", "mean"),
        평균소방용수거리_km=("소방용수거리_km", "mean"),
        평균경사도=("경사도(도)", "mean"),
        평균고도_m=("고도(m)", "mean"),
    )
    .reset_index()
)

vulnerable_grid = (
    grid_summary[grid_summary["산불건수"] >= 30]
    .sort_values(["평균취약점수", "산불건수"], ascending=[False, False])
    .head(20)
)

grid_summary.to_csv(OUT_DIR / "integrated_accessibility_grid_summary.csv", index=False, encoding="utf-8-sig")
vulnerable_grid.to_csv(OUT_DIR / "integrated_accessibility_vulnerable_grid_top20.csv", index=False, encoding="utf-8-sig")

vulnerable_grid.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
sc = ax.scatter(
    grid_summary["lon_grid"] + grid_size_deg / 2,
    grid_summary["lat_grid"] + grid_size_deg / 2,
    c=grid_summary["평균취약점수"],
    s=np.clip(grid_summary["산불건수"], 10, 350),
    cmap="YlOrRd",
    alpha=0.78,
)
plt.colorbar(sc, ax=ax, label="평균 진화 접근 취약 점수")
ax.set_title("격자별 산불 건수와 진화 접근 취약 점수")
ax.set_xlabel("경도")
ax.set_ylabel("위도")
plt.tight_layout()
plt.savefig(OUT_DIR / "integrated_accessibility_grid_map.png", dpi=180)
plt.show()

## 8. 지도 시각화: 취약 점수

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
ax.scatter(road_points["경도"], road_points["위도"], s=0.15, c="#94a3b8", alpha=0.16, label="임도")
ax.scatter(stations["경도"], stations["위도"], s=10, c="#2563eb", alpha=0.55, label="소방서")
ax.scatter(forest_ext["경도"], forest_ext["위도"], s=16, c="#16a34a", alpha=0.7, label="산불소화시설")
sc = ax.scatter(
    access["경도"],
    access["위도"],
    c=access["진화접근취약점수"],
    s=3,
    cmap="magma_r",
    alpha=0.7,
)
plt.colorbar(sc, ax=ax, label="진화 접근 취약 점수")
ax.set_title("산불 발생 지점별 진화 접근 취약 점수")
ax.set_xlabel("경도")
ax.set_ylabel("위도")
ax.legend(markerscale=2, loc="lower right")
plt.tight_layout()
plt.savefig(OUT_DIR / "integrated_accessibility_fire_map.png", dpi=180)
plt.show()

## 9. 인사이트 자동 요약

In [ ]:
n = len(access)
high_count = access["접근취약등급"].isin(["높음", "매우 높음"]).sum()
very_high_count = (access["접근취약등급"] == "매우 높음").sum()
road_far = (access["임도거리_km"] >= 10).sum()
station_far = (access["소방서거리_km"] >= 20).sum()
water_far = (access["소방용수거리_km"] >= 5).sum()
hard_terrain = ((access["경사도(도)"] >= 15) & (access["고도(m)"] >= 300)).sum()
complex_vulnerable = (
    (access["임도거리_km"] >= 10)
    & (access["소방서거리_km"] >= 20)
    & (access["소방용수거리_km"] >= 5)
).sum()

print(f"- 분석 대상 산불 지점은 {n:,}건입니다.")
print(f"- 진화 접근 취약 점수 평균은 {access['진화접근취약점수'].mean():.1f}점, 중앙값은 {access['진화접근취약점수'].median():.1f}점입니다.")
print(f"- 접근 취약 등급이 '높음' 이상인 산불 지점은 {high_count:,}건({high_count / n * 100:.1f}%)입니다.")
print(f"- 접근 취약 등급이 '매우 높음'인 산불 지점은 {very_high_count:,}건({very_high_count / n * 100:.1f}%)입니다.")
print(f"- 임도에서 10km 이상 떨어진 산불 지점은 {road_far:,}건({road_far / n * 100:.1f}%)입니다.")
print(f"- 소방서에서 20km 이상 떨어진 산불 지점은 {station_far:,}건({station_far / n * 100:.1f}%)입니다.")
print(f"- 소방용수시설에서 5km 이상 떨어진 산불 지점은 {water_far:,}건({water_far / n * 100:.1f}%)입니다.")
print(f"- 경사도 15도 이상이면서 고도 300m 이상인 지점은 {hard_terrain:,}건({hard_terrain / n * 100:.1f}%)입니다.")
print(f"- 임도/소방서/소방용수가 모두 먼 복합 취약 지점은 {complex_vulnerable:,}건({complex_vulnerable / n * 100:.1f}%)입니다.")

### 보고서에 쓸 수 있는 문장

- 산불 진화 접근성은 임도 거리만으로 판단하기 어렵기 때문에, 소방서 거리, 소방용수시설 거리, 산불소화시설 거리, 경사도, 고도를 함께 고려했다.
- 임도와 소방서가 모두 먼 산불 지점은 초기 출동 이후 실제 현장 접근 시간이 길어질 가능성이 있다.
- 소방용수시설과 먼 산불 지점은 진화 장비가 접근하더라도 물 공급 제약이 발생할 수 있다.
- 경사가 급하고 고도가 높은 지역은 장비 접근과 인력 이동이 모두 어려워 진화 난이도가 높을 가능성이 있다.
- 종합 취약 점수가 높은 격자는 임도 확충, 소방용수시설 보강, 산불소화시설 배치 검토의 우선 후보로 활용할 수 있다.